## Lasso Feature Selection

Before adding `SelectFromModel` to the final pipeline, we want to understand **which features Lasso keeps and which it zeros out**. This notebook is purely investigative — we fit the pipeline up to the encoding step, run LassoCV, and read the coefficients. The findings then justify adding `SelectFromModel(LassoCV)` to the final pipeline.

In [32]:
import numpy as np
import pandas as pd

import sys
sys.path.append("..")

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, TargetEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.compose import make_column_selector
from sklearn.linear_model import LassoCV

from src.preprocessing import AmesPreprocessor

In [33]:
df = pd.read_csv(filepath_or_buffer="../data/raw/AmesHousing.csv")


# drop the outliers - big houses sold for small price
outlier_mask = (df["Gr Liv Area"] > 4000) & (np.log1p(df["SalePrice"]) <= 12.5)
df = df[~outlier_mask].reset_index(drop=True)

# create target and separate predictions from features
y = np.log1p(df["SalePrice"])
df = df.drop(columns=["SalePrice"])

X_train = df.iloc[0:int(0.8 * len(df))]
X_test = df.iloc[int(0.8 * len(df)):len(df)]

y_train = y[0:int(0.8 * len(y))]
y_test = y[int(0.8 * len(y)):len(y)]

In [34]:
ordinal_cols = [
    "Kitchen Qual",
    "Bsmt Qual",
    "Bsmt Cond",
    "Exter Qual",
    "Garage Qual",
    "Garage Cond",
    "Garage Finish",
    "Bsmt Exposure",
    "BsmtFin Type 1",
    "BsmtFin Type 2",
    ]

ordinal_categories = [
    ["None", "Po", "Fa", "TA", "Gd", "Ex"],  # Kitchen Qual
    ["None", "Po", "Fa", "TA", "Gd", "Ex"],  # Bsmt Qual
    ["None", "Po", "Fa", "TA", "Gd", "Ex"],  # Bsmt Cond
    ["Po", "Fa", "TA", "Gd", "Ex"],          # Exter Qual
    ["None", "Po", "Fa", "TA", "Gd", "Ex"],  # Garage Qual
    ["None", "Po", "Fa", "TA", "Gd", "Ex"],  # Garage Cond
    ["None", "Unf", "RFn", "Fin"],           # Garage Finish
    ["None", "No", "Mn", "Av", "Gd"],        # Bsmt Exposure
    ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],  # BsmtFin Type 1
    ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],  # BsmtFin Type 2
]

target_encode_cols = ["Neighborhood"]

one_hot_cols = [
    "House Style",
    "Garage Type",
    "Mas Vnr Type",
    "Electrical",
]

column_transformer = ColumnTransformer(
    transformers=[
        ("numerical", StandardScaler(), make_column_selector(dtype_include=np.number)),
        ("one_hot", OneHotEncoder(handle_unknown="ignore"), one_hot_cols),
        ("target", TargetEncoder(), target_encode_cols),
        ("ordinal", OrdinalEncoder(categories=ordinal_categories, handle_unknown="use_encoded_value", unknown_value=-1), ordinal_cols)
    ]
)

pipeline = Pipeline(
    steps=[
        ("preprocessing", AmesPreprocessor()),
        ("transformer", column_transformer)
    ]
)

pipeline.fit(X=X_train, y=y_train)
transformed_df = pipeline.transform(X=X_train)

# print features after the fiture engineering and encodings 
print(column_transformer.get_feature_names_out())

['numerical__Order' 'numerical__PID' 'numerical__MS SubClass'
 'numerical__Lot Frontage' 'numerical__Lot Area' 'numerical__Overall Qual'
 'numerical__Overall Cond' 'numerical__Year Remod/Add'
 'numerical__Mas Vnr Area' 'numerical__BsmtFin SF 1'
 'numerical__BsmtFin SF 2' 'numerical__Bsmt Unf SF'
 'numerical__Total Bsmt SF' 'numerical__2nd Flr SF'
 'numerical__Low Qual Fin SF' 'numerical__Gr Liv Area'
 'numerical__Bsmt Full Bath' 'numerical__Bsmt Half Bath'
 'numerical__Full Bath' 'numerical__Half Bath' 'numerical__Bedroom AbvGr'
 'numerical__Kitchen AbvGr' 'numerical__Fireplaces'
 'numerical__Garage Area' 'numerical__Wood Deck SF'
 'numerical__Open Porch SF' 'numerical__Enclosed Porch'
 'numerical__3Ssn Porch' 'numerical__Screen Porch' 'numerical__Pool Area'
 'numerical__Misc Val' 'numerical__Mo Sold' 'numerical__Yr Sold'
 'numerical__HasBasement' 'numerical__HasGarage' 'numerical__Has2ndFloor'
 'numerical__IsRemodeled' 'numerical__HouseAge' 'numerical__RemodelAge'
 'numerical__TotalSF

In [35]:
lasso = LassoCV(cv=5, max_iter=10000)
lasso.fit(X=transformed_df, y=y_train)

print(lasso.alpha_)

0.0006361606196821411


### Optimal Alpha

LassoCV tried many alpha values across 5 folds and selected the one with the lowest mean MSE. The selected alpha is our minimum contribution threshold — any feature whose coefficient absolute value is below this threshold gets zeroed out.

In [37]:
coef_df = pd.DataFrame({
    "feature": column_transformer.get_feature_names_out(),
    "coefficient": lasso.coef_
})

# sort by absolute magnitude (largest impact first)
coef_df["abs_coef"] = coef_df["coefficient"].abs()
coef_df = coef_df.sort_values(by="abs_coef", ascending=False).drop(columns=["abs_coef"]).reset_index(drop=True)

# mask and filter
is_zero_mask = np.isclose(coef_df["coefficient"], 0, atol=1e-5)
selected_features_df = coef_df[~is_zero_mask]   # keep non-zeroed features
dropped_features = coef_df[is_zero_mask]


print(f"Features before Lasso: {len(coef_df)}")
print(f"Features kept: {len(selected_features_df)}")
print(f"Features dropped: {len(dropped_features)}")

print("\nDropped features:")
print(dropped_features["feature"].to_string(index=False))

Features before Lasso: 76
Features kept: 48
Features dropped: 28

Dropped features:
   one_hot__House Style_SLvl
   one_hot__Electrical_FuseP
   numerical__Year Remod/Add
        numerical__Pool Area
     numerical__Mas Vnr Area
        ordinal__Garage Qual
      numerical__Has2ndFloor
      numerical__Bsmt Unf SF
    numerical__Total Bsmt SF
       numerical__2nd Flr SF
 one_hot__House Style_2.5Fin
   one_hot__Electrical_SBrkr
     one_hot__Electrical_Mix
   one_hot__Electrical_FuseF
 one_hot__Garage Type_2Types
   one_hot__Electrical_FuseA
 one_hot__House Style_2.5Unf
    numerical__Bedroom AbvGr
one_hot__Mas Vnr Type_CBlock
 one_hot__House Style_SFoyer
one_hot__Mas Vnr Type_BrkCmn
   one_hot__Garage Type_None
          numerical__Yr Sold
one_hot__Garage Type_CarPort
one_hot__Garage Type_BuiltIn
one_hot__Garage Type_Basment
 one_hot__Garage Type_Attchd
 one_hot__House Style_1.5Unf


### Results

LassoCV selected **alpha = 0.00064**, which zeroed out **28 features** out of **76 total**, leaving **48 features** for the final model.

The dropped features fall into clear groups that all make intuitive sense:

**Electrical system** — all 5 categories were zeroed out entirely. Almost every house in Ames uses standard circuit breakers — the other types are so rare they carry no reliable signal.

**Garage Type** — nearly all categories zeroed out. Whether the garage is attached, built-in, basement or carport doesn't drive price. What matters is garage size, which is already captured by `Garage Area`.

**House Style rare categories** — `2.5Fin`, `2.5Unf`, `SFoyer`, `SLvl`, `1.5Unf` all zeroed. These styles have very few houses, not enough data for Lasso to justify keeping a non-zero weight.

**`Total Bsmt SF` and `2nd Flr SF`** — both zeroed because `TotalSF` we created during feature engineering already combines basement and above-ground area. Lasso correctly identified the redundancy — these columns add no new information on top of `TotalSF`.

**`Yr Sold`** — year of sale has no meaningful price signal after controlling for house age and quality. The market didn't shift dramatically enough across the years in this dataset to matter.

**`Bedroom AbvGr`** — number of bedrooms is redundant with `Gr Liv Area`. More rooms naturally means more area, and the model prefers the continuous area measurement over the discrete room count.

**`Year Remod/Add`** — already captured by `RemodelAge` and `IsRemodeled` which we engineered. The raw year adds nothing once the age-based features exist.

The feature selection confirms that our feature engineering decisions during preprocessing were correct — Lasso independently arrived at the same conclusions we reasoned through manually in EDA.